# 🔬 Modül 4: LangSmith & Telemetri Atölyesi (Gözlemlenebilirlik)

Bu interaktif not defterinde, çoklu ajan sistemlerinin kör noktalarını ortadan kaldıran **gözlemlenebilirlik (observability)**, **telemetri**, **gecikme (latency)** ve **maliyet (token/cost) analizini** adım adım, canlı kodlar ve kişisel not alanlarıyla deneyimleyeceğiz.

### 🗺️ 5 Aşamalı Atölye Yol Haritası:
```
┌────────────────────────────────────────────────────────────────────────┐
│ BÖLÜM 1: Gözlemlenebilirlik Nedir? (Teori & Not Alanı)                 │
├────────────────────────────────────────────────────────────────────────┤
│ BÖLÜM 2: API Key Havuzu & LangSmith Çevre Değişkenleri Kurulumu        │
├────────────────────────────────────────────────────────────────────────┤
│ BÖLÜM 3: Çok Araçlı Ajanın Koşturulması (Live Run & Callbacks)         │
├────────────────────────────────────────────────────────────────────────┤
│ BÖLÜM 4: Token Tüketimi, Gecikme (Latency) ve Maliyet Tablosu          │
├────────────────────────────────────────────────────────────────────────┤
│ BÖLÜM 5: Run Tree İnceleme & Deney Gözlem Günlüğü (Kişisel Notlar)     │
└────────────────────────────────────────────────────────────────────────┘
```

> **💡 Nasıl Kullanılır?**
> Her kod hücresini sırayla `Shift + Enter` ile çalıştırabilirsiniz. Kod hücrelerinin hemen altındaki **📝 Benim Notlarım** alanlarına çift tıklayarak kendi gözlemlerinizi, sorularınızı ve projelerinize dair fikirlerinizi kaydedebilirsiniz.

---
## 🏛️ BÖLÜM 1: Gözlemlenebilirlik (Observability) Nedir? Neden Şarttır?

Geleneksel yazılımlarda loglama (`print`, `logging`, `ELK`), fonksiyon çağrılarının giriş-çıkış değerlerini ve hata yığınlarını (stack trace) takip etmek için yeterliydi. Ancak **Otonom LLM Ajanları** deterministik değil, **olasılıksal (probabilistic)** çalışır:

1. **Bilişsel Takip (Cognitive Tracing):** Ajan neden belirli bir aracı seçti? Hangi prompt talimatı veya araç açıklaması bu kararı tetikledi?
2. **Kör Uçuş Tehlikesi:** Çoklu ajan veya supervisor yapılarında birden fazla uzman ajan çalışırken, biri halüsinasyon gördüğünde veya döngüye (infinite loop) girdiğinde sorunun kaynağını bulmak telemetri olmadan imkansızdır.
3. **Maliyet & Darboğaz (Bottleneck) Analizi:** Hangi düğüm kaç milisaniye harcadı? Hangi LLM çağrısı gereksiz token tüketip bütçeyi eritti?
4. **Run Tree (Çalışma Ağacı) Hiyerarşisi:** LangSmith, tek bir kullanıcı sorgusuyla başlayan tüm süreci hiyerarşik bir ağaç yapısında kayıt altına alır:
   - `Parent Run` (StateGraph Invoke)
     - `Child Run` (Agent Düğümü - LLM Kararı)
     - `Tool Runs` (audit_system_logs, calculate_cost_projection)
     - `Child Run` (Agent Düğümü - Nihai Sentez ve Cevap)

### 📝 Bölüm 1 - Benim Notlarım & Gözlemlerim
> *(Bu alana çift tıklayarak kendi notlarınızı yazabilirsiniz)*

- **Burada öğrendiğim en kritik şey:** ...
- **Şirketimde / Kendi projelerimde kullanabileceğim senaryolar:** ...
- **Kafama takılan / derinleşmek istediğim sorular:** ...
---

## ⚙️ BÖLÜM 2: 5'li Key Havuzu & LangSmith Çevre Değişkenleri Kurulumu

Bu hücrede:
1. Proje kök dizinindeki `model_factory.py` modülü içeri aktarılır.
2. Sabit **`gemini-3.8-flash`** modeli ve `.env` dosyasındaki API anahtarı havuzu (`GOOGLE_API_KEYS`) doğrulanır.
3. LangSmith izleme bayrakları (`LANGCHAIN_TRACING_V2`, `LANGCHAIN_PROJECT`, `LANGCHAIN_API_KEY`) dinamik olarak denetlenir.
   - Eğer geçerli bir `LANGCHAIN_API_KEY` varsa bulut dashboard'u (`smith.langchain.com`) devreye alınır.
   - Yoksa sistem 401 hatası almamak için güvenle yerel (local) telemetri moduna geçer.

In [ ]:
# 1. Çalışma dizini ve model_factory entegrasyonu
import os
import sys
from dotenv import load_dotenv
from rich.console import Console
from rich.panel import Panel
from rich.table import Table

# Proje ana dizinini sys.path'e ekleyelim
current_dir = os.path.abspath("")
parent_dir = os.path.dirname(current_dir) if "04_gozlemlenebilirlik" in current_dir else current_dir
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

load_dotenv(os.path.join(parent_dir, ".env"))
console = Console()

# Model fabrikasından sabit modeli ve anahtar havuzunu yükle
from model_factory import get_gemini_llm, TARGET_MODEL, API_KEY_POOL

# 2. LangSmith Ortam Değişkenleri Kontrolü
langsmith_key = os.getenv("LANGCHAIN_API_KEY", "").strip()
cloud_tracing_active = bool(langsmith_key and langsmith_key.startswith("lsv2_"))
project_name = os.getenv("LANGCHAIN_PROJECT", "otonom-ajan-sirketi")

if cloud_tracing_active:
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_PROJECT"] = project_name
    os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
    status_text = f"[bold green]✔ LangSmith Cloud Tracing AKTİF![/bold green]\n👉 Dashboard: https://smith.langchain.com/o/default/projects/p/{project_name}"
else:
    os.environ["LANGCHAIN_TRACING_V2"] = "false"
    status_text = "[bold yellow]ℹ LANGCHAIN_API_KEY bulunamadı / yerel telemetri aktif.[/bold yellow]\n(Bulut grafiği için https://smith.langchain.com üzerinden ücretsiz API anahtarı ekleyebilirsiniz.)"

# Durum Özeti Kartı
summary = Table.grid(padding=1)
summary.add_column(style="bold cyan")
summary.add_column(style="white")
summary.add_row("Aktif Sabit Model:", TARGET_MODEL)
summary.add_row("API Key Havuzu:", f"{len(API_KEY_POOL)} adet anahtar yüklendi (Otomatik Rotasyon Hazır)")
summary.add_row("LangSmith Durumu:", status_text)

console.print(Panel(summary, title="🚀 Sistem & Gözlemlenebilirlik Kurulumu", border_style="green"))

### 📝 Bölüm 2 - Benim Notlarım: API Havuzu ve İzleme Ortamı
> *(API anahtarı havuzu, failover veya LangSmith bağlantınız ile ilgili notlarınızı buraya ekleyin)*

- **Key Havuzu Rotasyonu Gözlemim:** ...
- **LangSmith Proje Yapılandırması Notum:** ...
---

## 🤖 BÖLÜM 3: Çok Araçlı Ajanın Koşturulması (Live Run)

Bu hücrede operasyonel bir ajan senaryosu çalıştıracağız:
1. **Araçlar (Tools):**
   - `audit_system_logs`: Belirtilen mikroservisin sağlık, hata, CPU ve bellek durumunu denetler (0.3 sn ağ gecikmesi simülasyonu).
   - `calculate_cost_projection`: Aylık gider ve enflasyon faktörüyle kümülatif maliyet projeksiyonu hesaplar (0.4 sn ağ gecikmesi simülasyonu).
2. **Canlı Callback İzleyici (`LiveMetricsCallbackHandler`):** Her LLM düşünme ve Tool çalışma adımını anında konsola yansıtır.
3. **StateGraph & Kalıcılık:** Ajan `SqliteSaver` ile `checkpoints.db`'ye kaydedilir.
4. **Kurumsal Metadata & Tags:** Sorgu `tags=["production", "q4-audit", "infra-team"]` ve kurumsal metadata ile zenginleştirilir.

In [ ]:
import time
import sqlite3
from typing import Annotated
from typing_extensions import TypedDict

from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.tools import tool
from langchain_core.callbacks.base import BaseCallbackHandler
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.sqlite import SqliteSaver

# 1. Canlı Telemetri Callback Sınıfı
class LiveMetricsCallbackHandler(BaseCallbackHandler):
    """Her adımın süresini, çağrılan araçları ve gecikmeleri anlık bildiren izleyici."""
    def on_llm_start(self, serialized, prompts, **kwargs):
        console.print("  ⚡ [bold yellow][LLM Başladı][/bold yellow]: Model muhakeme yapıyor...")

    def on_tool_start(self, serialized, input_str, **kwargs):
        name = serialized.get("name", "Bilinmeyen Araç")
        console.print(f"  🔧 [bold cyan][Araç Başladı][/bold cyan]: {name} çağrıldı...")

    def on_tool_end(self, output, **kwargs):
        console.print("  ✔ [bold green][Araç Tamamlandı][/bold green]: Çıktı state'e aktarıldı.")

# 2. Simüle Edilmiş Araçlar (Gecikmeli & Gerçekçi)
@tool
def calculate_cost_projection(current_expense: float, months: int) -> str:
    """Belirli bir aylık gider için geleceğe dönük kümülatif maliyet projeksiyonu hesaplar."""
    time.sleep(0.4)  # Ağ ve hesaplama gecikmesi simülasyonu
    total = current_expense * months * 1.08  # %8 enflasyon ve büyüme payı
    return f"{months} aylık tahmini toplam maliyet projeksiyonu: {total:,.2f} TL"

@tool
def audit_system_logs(service_name: str) -> str:
    """Belirtilen mikroservisin sistem sağlık ve hata loglarını denetler."""
    time.sleep(0.3)  # I/O log okuma gecikmesi simülasyonu
    return f"[{service_name.upper()}]: Sistem sağlıklı, 0 kritik hata, CPU: %14, Bellek: 420MB."

tools = [calculate_cost_projection, audit_system_logs]

# 3. Modeli Araçlarla Donat
llm_with_tools = get_gemini_llm(temperature=0.2, tools=tools)

# 4. Graf & State Tanımı
class ObservabilityState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

def agent_node(state: ObservabilityState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

workflow = StateGraph(ObservabilityState)
workflow.add_node("agent", agent_node)
workflow.add_node("tools", ToolNode(tools=tools))

workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", tools_condition)
workflow.add_edge("tools", "agent")

# Kalıcılık (Checkpointer)
db_path = os.path.join(current_dir, "checkpoints.db")
conn = sqlite3.connect(db_path, check_same_thread=False)
memory = SqliteSaver(conn)
app = workflow.compile(checkpointer=memory)

# 5. Canlı Çalıştırma
soru = (
    "Finans mikroservisimizin sağlık durumunu denetle. "
    "Ayrıca aylık 85.000 TL olan sunucu giderimiz için 6 aylık projeksiyon çıkar."
)

console.print(f"\n[bold white]Kullanıcı Talebi:[/bold white] {soru}\n")

config = {
    "configurable": {"thread_id": "atolye-oturum-001"},
    "tags": ["production", "q4-audit", "infra-team", "workshop-run"],
    "metadata": {
        "environment": "production",
        "cost_center": "Engineering-Ops",
        "triggered_by": "Jupyter-Workshop"
    },
    "callbacks": [LiveMetricsCallbackHandler()]
}

start_time = time.time()
result = app.invoke({"messages": [HumanMessage(content=soru)]}, config=config)
total_latency = round(time.time() - start_time, 3)

console.print(f"\n[bold green]✔ Çalışma tamamlandı! (Toplam Süre: {total_latency}s)[/bold green]")

### 📝 Bölüm 3 - Benim Notlarım: Ajanın Karar Alma Süreci
> *(Ajanın karar adımları, araç seçme sırası ve callback çıktıları ile ilgili gözlemleriniz)*

- **Ajanın Karar Alma Mantığı (Hangi araçları neden seçti?):** ...
- **Adımların Sıralaması ve Senkronizasyonu:** ...
---

## 📊 BÖLÜM 4: Token Tüketimi, Gecikme (Latency) ve Maliyet Tablosu

Ajan bir görevi tamamlarken arka planda kaç mesaj üretildi, kaç token harcandı ve bize maliyeti ne oldu?
Aşağıdaki hücrede:
1. State içerisindeki tüm `HumanMessage`, `AIMessage` ve `ToolMessage` blokları ayrıştırılır.
2. Yaklaşık token hesabı ve Gemini 3.8 Flash kurumsal tarifesine göre tahmini maliyet ($USD) hesaplanır.
3. `rich` kütüphanesiyle şık bir telemetri tablosu ve ajan nihai raporu basılır.

In [ ]:
def get_text(content) -> str:
    if isinstance(content, str):
        return content
    elif isinstance(content, list):
        return " ".join([b.get("text", "") if isinstance(b, dict) else str(b) for b in content])
    return str(content)

# Metrik Hesaplamaları
messages = result["messages"]
message_count = len(messages)
total_chars = sum(len(get_text(m.content)) for m in messages)
est_tokens = int(total_chars / 4)  # Ortalama 4 karakter ~ 1 token
est_cost = (est_tokens / 1000) * 0.0001  # Gemini 3.8 Flash ~$0.0001 / 1K token

# Rich Telemetri Tablosu
table = Table(title="📊 Graf Çalışma ve Gözlemlenebilirlik Metrikleri", border_style="green")
table.add_column("Metrik Adı", style="bold cyan", no_wrap=True)
table.add_column("Değer", style="bold magenta")
table.add_column("Açıklama", style="dim")

table.add_row("Toplam Süre (Latency)", f"{total_latency} saniye", "Tüm LLM muhakeme + Tool I/O süreleri")
table.add_row("Üretilen Mesaj Adımı", str(message_count), "Human -> AI (ToolCall) -> Tools -> AI (Rapor)")
table.add_row("Tahmini Harcanan Token", f"~{est_tokens:,} token", "Prompt + Tool yanıtları + Completion")
table.add_row("Tahmini İşlem Maliyeti", f"${est_cost:.6f} USD", "Gemini 3.8 Flash tarifesiyle yaklaşık harcama")
table.add_row("Kalıcılık Katmanı", "SQLite (checkpoints.db)", "thread_id: atolye-oturum-001 altına kaydedildi")

console.print(table)

# Nihai Ajan Yanıtı
final_reply = get_text(messages[-1].content)
console.print("\n[bold green]=== AJANIN NİHAİ RAPORU ===[/bold green]")
console.print(Panel(final_reply, title="Ajan Çıktısı", border_style="blue"))

### 📝 Bölüm 4 - Benim Notlarım: Maliyet & Performans Analizi
> *(Gecikme süresi, token sarfiyatı ve maliyet tablosuna dair çıkarımlarınız)*

- **Gecikme (Latency) Değerlendirmem:** ...
- **Token ve Maliyet Verimliliği:** ...
---

## 🔍 BÖLÜM 5: Run Tree İnceleme & Deney Gözlem Günlüğü

### 🌲 LangSmith Run Tree (Çalışma Ağacı) Anatomisi:
Eğer `LANGCHAIN_API_KEY` eklediyseniz, [smith.langchain.com](https://smith.langchain.com) adresindeki projenizde bu çalışmayı hiyerarşik bir ağaç olarak görürsünüz:

```text
Root Run: LangGraph Workflow (RunnableSequence) [~1.8s]
├── Node: agent (ChatGoogleGenerativeAI) [~0.9s]
│   └── Input: "Finans mikroservisimizin sağlık..."
│   └── Output: tool_calls=[calculate_cost_projection, audit_system_logs]
├── Node: tools (ToolNode) [~0.7s]
│   ├── Tool: audit_system_logs [~0.3s]
│   └── Tool: calculate_cost_projection [~0.4s]
└── Node: agent (ChatGoogleGenerativeAI) [~0.8s]
    └── Input: [Original Prompt + Tool Results]
    └── Output: "Finans mikroservisimizin sağlık durumu..."
```

### 💡 Gözlemlenebilirlikten Çıkarılan Temel Kazanımlar:
1. **Bilişsel Şeffaflık:** LLM'in prompt içine enjekte edilen sistem mesajlarını ve araç tanımlarını nasıl algıladığını karakter karakter görebilirsiniz.
2. **Hata İzolasyonu:** Eğer bir araç başarısız olursa veya beklenmeyen format dönerse, hangi milisaniyede hangi alt düğümün çöktüğü anında tespit edilir.
3. **Maliyet Bütçeleme (Cost Guardrails):** Şirketinizde binlerce sorgu çalışırken hangi departmanın veya özelliğin kota aştığı doğrudan etiketler (`tags`, `metadata`) üzerinden filtrelenir.

---
### 🔍 Bu Görevden Çıkardığım Dersler & Kişisel Deney Günlüğüm
*(Atölyeyi tamamladıktan sonra buraya kendi değerlendirmelerinizi not alınız)*

1. **Gözlem:** ...
2. **Darboğaz Noktası:** ...
3. **Optimizasyon Fikrim:** ...
4. **Otonom Şirket Mimarisine Entegrasyon:** ...